<a href="https://colab.research.google.com/github/Adhira-Deogade/pytorch-learnings/blob/main/notebooks/sd/sd_deepinv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepinv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.1/724.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.0/983.0 kB 69.4 MB/s eta 0:00:00


In [2]:
import torch
import deepinv
from torchvision import datasets, transforms

In [3]:
batch_size, image_size = 32, 32

In [4]:
train_transforms = transforms.Compose(
    [
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.0,),
            (1.0,)
        )
    ]
)

In [5]:
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        root='./data',
        train=True,
        download=True,
        transform=train_transforms
    ),
    batch_size=batch_size,
    shuffle=True
)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.63MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 136kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]


In [6]:
print(train_loader)

In [7]:
for batch, (features, labels) in enumerate(train_loader):
    print(batch, features.shape, labels)
    break

0 torch.Size([32, 1, 32, 32]) tensor([9, 3, 4, 8, 4, 9, 7, 6, 2, 0, 1, 9, 2, 6, 9, 8, 6, 0, 2, 6, 4, 1, 0, 4,
        2, 3, 8, 3, 5, 3, 9, 0])


In [8]:
lr = 1e-4
epochs = 100

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [11]:
model = deepinv.models.DiffUNet(
    in_channels=1,
    out_channels=1,
    pretrained=None
).to(device)
print(model)

DiffUNet(
  (time_embed): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): SiLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
  )
  (input_blocks): ModuleList(
    (0): TimestepEmbedSequential(
      (0): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (1): TimestepEmbedSequential(
      (0): ResBlock(
        (in_layers): Sequential(
          (0): GroupNorm32(32, 128, eps=1e-05, affine=True)
          (1): SiLU()
          (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (h_upd): Identity()
        (x_upd): Identity()
        (emb_layers): Sequential(
          (0): SiLU()
          (1): Linear(in_features=512, out_features=256, bias=True)
        )
        (out_layers): Sequential(
          (0): GroupNorm32(32, 128, eps=1e-05, affine=True)
          (1): SiLU()
          (2): Dropout(p=0.1, inplace=False)
          (3): Conv2d(128, 128, kernel_size=(3, 3), strid

In [12]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=lr
)

In [14]:
mse = deepinv.loss.MSE()

In [15]:
beta_start = 1e-4
beta_end = 0.02
timesteps = 1000

In [16]:
betas = torch.linspace(
    beta_start,
    beta_end,
    timesteps,
    device=device
)

In [17]:
# Plot betas later

In [18]:
alphas = 1.0 - betas
alphas_cumpord = torch.cumprod(alphas, dim=0) # dim=0 means along columns
sqrt_aphas_cumprod = torch.sqrt(alphas_cumpord)
srqt_one_minus_alphas_cumprod = torch.sqrt(
    1.0 - alphas_cumpord
)

In [ ]:
for epoch in range(epochs):
  model.train()
   # we don't care about labels
  for data, _ in train_loader:
    imgs = data.to(device)
    noise = torch.rand_like(imgs)
    t = torch.randint(
        0,
        timesteps,
        (imgs.shape[0],),
        device=device
    )
    noised_image = (sqrt_aphas_cumprod[t, None, None, None] * imgs) + (srqt_one_minus_alphas_cumprod[t, None, None, None] * noise)

    # gradient descent
    optimizer.zero_grad()
    pred_noise = model(noised_image, t, type_t='timestep')
    loss = mse(pred_noise, noise).mean()
    loss.backward()
    optimizer.step()
torch.save(model.state_dict(), 'trained_diffusion_model.pth')